In [2]:
import os
import json
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

In [3]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR = CONFIGS['filepaths']['splits']
PREDSDIR  = CONFIGS['filepaths']['predictions']
EQUATIONS = CONFIGS['experiments']['sr']['optimizedeqs']
SPLIT     = 'test'
LANDFRAC  = 0.5
NHOURBINS = 8
MODELS    = ['sr_atm_eq','sr_sfc_eq','sr_all_eq','sr_all_pc_eq']
REGIONS   = ['Ocean','Land']

In [4]:
def bin_by_hour(z,mask):
    counts = np.bincount(hourbin[mask],minlength=NHOURBINS)
    sums   = np.bincount(hourbin[mask],weights=z[mask],minlength=NHOURBINS)
    return sums/np.maximum(counts,1)

In [5]:
with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    ntime,nlat,nlon = (ds.sizes[dim] for dim in ['time','lat','lon'])
    lon   = ds['lon'].values
    hours = ds['time'].dt.hour.values
    tp    = ds['tp'].transpose('time','lat','lon').values.ravel()
    lf    = xr.broadcast(ds['lf'],ds['tp'])[0].transpose('time','lat','lon').values.ravel()
alltp = {'ERA5':tp}
for name in MODELS:
    with xr.open_dataset(os.path.join(PREDSDIR,f'{name}_{SPLIT}_predictions.nc')) as ds:
        alltp[EQUATIONS[name]['description']] = ds['tp'].squeeze().transpose('time','lat','lon').values.ravel()

finite      = np.all([np.isfinite(values) for values in alltp.values()],axis=0)
localhour   = (hours[:,None,None]+lon[None,None,:]/15.0)%24
hourbin     = (np.broadcast_to(localhour,(ntime,nlat,nlon)).ravel()//(24/NHOURBINS)).astype(int)
hourcenters = (np.arange(NHOURBINS)+0.5)*24/NHOURBINS
regionmasks = {'Ocean':finite&(lf<LANDFRAC),'Land':finite&(lf>=LANDFRAC)}
cycles      = {(name,region):bin_by_hour(values,regionmasks[region]) for name,values in alltp.items() for region in REGIONS}

rows = []
for name,values in alltp.items():
    row = {'Model':name}
    for region in REGIONS:
        mask = regionmasks[region]
        peak = hourcenters[np.argmax(cycles[name,region])]
        row[region,r'$\mathrm{R}^2$']         = np.nan if name=='ERA5' else 1-np.sum((values[mask]-tp[mask])**2)/np.sum((tp[mask]-tp[mask].mean())**2)
        row[region,'Range (%)']  = 100*np.ptp(cycles[name,region])/np.ptp(cycles['ERA5',region])
        row[region,'$r$']          = np.nan if name=='ERA5' else np.corrcoef(cycles[name,region],cycles['ERA5',region])[0,1]
        row[region,'Peak (LST)'] = f'{int(peak-12/NHOURBINS):02d}--{int(peak+12/NHOURBINS):02d}'
    rows.append(row)
table = pd.DataFrame(rows).set_index('Model')
table.columns = pd.MultiIndex.from_tuples(table.columns)

FileNotFoundError: [Errno 2] No such file or directory: '/global/cfs/cdirs/m4334/sferrett/monsoon-discovery/data/predictions/sr_sfc_eq_test_predictions.nc'

In [6]:
formats = {r'$\mathrm{R}^2$':'{:.3f}','Range (%)':'{:.0f}','$r$':'{:.2f}','Peak (LST)':'{}'}
display(table.style.format({column:(lambda value,fmt=formats[column[1]]:'--' if isinstance(value,float) and np.isnan(value) else fmt.format(value)) for column in table.columns}))

NameError: name 'table' is not defined